# EXP/0413 — Negotiation Zone Analysis: `cooperative`

**Restriction**: Only transitions where $v \leq m_t \leq s_{t-1}$ (the negotiation zone).
The floor regime ($m_t < v$, seller ignores buyer) and acceptance regime
($m_t > s_{t-1}$, seller accepts) are excluded as their behavior is deterministic.

**Model under test** (within the zone):

$$s_t = \beta_0 + \beta_m \cdot m_t + \beta_s \cdot s_{t-1}$$

In fractional coordinates:

$$\text{frac}(s_t) = \beta_0 + \beta_m \cdot \text{frac}(m_t) + \beta_s \cdot \text{frac}(s_{t-1})$$

where $\text{frac}(x) = (x - v)/(\text{market} - v)$.

---

## S0. Setup

In [ ]:
import json, sys, glob, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 130
plt.rcParams['figure.figsize'] = (10, 5)
warnings.filterwarnings('ignore')

STRATEGY = "cooperative"
SLABEL = "Cooperative"
RESULTS_DIR = Path("results") / STRATEGY

def ols(X, y):
    n, p = X.shape
    b, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    yh = X @ b; r = y - yh
    rmse = float(np.sqrt(np.mean(r**2)))
    ss_r = float(np.sum(r**2)); ss_t = float(np.sum((y - np.mean(y))**2))
    r2 = 1 - ss_r/ss_t if ss_t > 1e-12 else 1.0
    dof = max(n-p,1); s2 = ss_r/dof
    try: se = np.sqrt(np.diag(s2 * np.linalg.inv(X.T @ X)))
    except: se = np.full(p, np.nan)
    return b, rmse, r2, yh, se

def load_data():
    trans = []
    for f in sorted(glob.glob(str(RESULTS_DIR / "*.json"))):
        if "transitions" in f: continue
        data = json.load(open(f))
        if "runs" not in data: continue
        for ri, run in enumerate(data["runs"]):
            meta = run["metadata"]; v=meta["v"]; cost=meta["product"]["cost"]; market=meta["product"]["market_price"]
            rounds = run["rounds"]; m_init=rounds[0]["buyer_price"]; s_init=rounds[0]["supplier_price"]
            for i in range(1, len(rounds)):
                prev=rounds[i-1]; curr=rounds[i]
                sp=prev["supplier_price"]; mc=curr["buyer_price"]; sc=curr["supplier_price"]
                if sp is None or sc is None: continue
                trans.append(dict(m_t=mc, s_prev=sp, s_t=sc, v=v, cost=cost, market=market,
                    round_t=curr["round"], buyer_type=meta["buyer_type"]["name"],
                    product=meta["product"]["name"], supplier_id=meta["supplier"]["id"],
                    m_init=m_init, s_init=s_init if s_init else np.nan,
                    m_prev=prev["buyer_price"], template_id=curr.get("template_id",-1),
                    s_prev2=rounds[i-2]["supplier_price"] if i>=2 else None, run_idx=ri))
    return trans

all_trans = load_data()

# ── ZONE FILTER ──
m_all = np.array([t["m_t"] for t in all_trans])
s_all = np.array([t["s_prev"] for t in all_trans])
v_all = np.array([t["v"] for t in all_trans])
zone_mask = (m_all >= v_all) & (m_all <= s_all)
trans = [t for t, keep in zip(all_trans, zone_mask) if keep]

m   = np.array([t["m_t"] for t in trans])
s   = np.array([t["s_prev"] for t in trans])
y   = np.array([t["s_t"] for t in trans])
v   = np.array([t["v"] for t in trans])
mkt = np.array([t["market"] for t in trans])
ti  = np.array([t["round_t"] for t in trans], dtype=float)
zone = mkt - v; zone[zone < 0.01] = 0.01
fm  = (m - v) / zone; fs = (s - v) / zone; fy = (y - v) / zone

print(f"Strategy: {SLABEL}")
print(f"All transitions: {len(all_trans)}")
print(f"Zone transitions (v <= m_t <= s_prev): {len(trans)} ({len(trans)/len(all_trans)*100:.0f}%)")
print(f"Products: {sorted(set(t['product'] for t in trans))}")
print(f"v values: {sorted(set(round(t['v'],2) for t in trans))}")
print(f"frac(m_t) range: [{fm.min():.3f}, {fm.max():.3f}]")
print(f"frac(s_prev) range: [{fs.min():.3f}, {fs.max():.3f}]")
print(f"frac(s_t) range: [{fy.min():.3f}, {fy.max():.3f}]")

---
## S1. Shape discovery within the negotiation zone

Since we have filtered to $v \leq m_t \leq s_{t-1}$, all points lie in the
region where the seller is genuinely negotiating.  We examine whether the
relationship is linear, and what the slope is.

### S1.1 Scatter: $s_t$ vs $m_t$ colored by $s_{t-1}$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Raw prices
ax = axes[0]
sc = ax.scatter(m, y, c=s, cmap='viridis', alpha=0.5, s=15, edgecolors='none')
plt.colorbar(sc, ax=ax, label="$s_{t-1}$")
lo = min(m.min(),y.min())*0.9; hi = max(m.max(),y.max())*1.05
ax.plot([lo,hi],[lo,hi],'k--',lw=0.8,alpha=0.3)
ax.set_xlabel("$m_t$ (\$)"); ax.set_ylabel("$s_t$ (\$)")
ax.set_title("Raw prices")

# Fractional
ax = axes[1]
sc = ax.scatter(fm, fy, c=fs, cmap='viridis', alpha=0.5, s=15, edgecolors='none')
plt.colorbar(sc, ax=ax, label="frac($s_{t-1}$)")
ax.plot([0,1],[0,1],'k--',lw=0.8,alpha=0.3)
ax.set_xlabel("frac($m_t$)"); ax.set_ylabel("frac($s_t$)")
ax.set_title("Fractional coordinates")

fig.suptitle(f"Fig 1: Zone-only scatter — {SLABEL} (N={len(m)})", y=1.01, fontsize=12)
fig.tight_layout(); plt.show()

### S1.2 Binned by $s_{t-1}$ with local OLS (fractional space)

In [ ]:
n_bins = 4
fs_edges = np.percentile(fs, np.linspace(0, 100, n_bins + 1))
fs_edges[-1] += 0.01
colors = ['#534AB7','#1D9E75','#D85A30','#378ADD','#D4537E']

fig, axes = plt.subplots(1, n_bins, figsize=(4*n_bins, 4.5))
if n_bins == 1: axes = [axes]

for bi in range(n_bins):
    ax = axes[bi]
    mask = (fs >= fs_edges[bi]) & (fs < fs_edges[bi+1])
    fmg, fyg = fm[mask], fy[mask]
    ax.scatter(fmg, fyg, alpha=0.5, s=12, c=colors[bi], edgecolors='none')

    if mask.sum() >= 4:
        Xg = np.column_stack([np.ones(mask.sum()), fmg])
        bg, _, r2g, _, _ = ols(Xg, fyg)
        xr = np.linspace(fmg.min(), fmg.max(), 50)
        ax.plot(xr, bg[0]+bg[1]*xr, c=colors[bi], lw=2,
                label=f"slope={bg[1]:.3f}, R$^2$={r2g:.3f}")

    ax.plot([0,1],[0,1],'k--',lw=0.7,alpha=0.3)
    ax.set_xlabel("frac($m_t$)"); ax.set_ylabel("frac($s_t$)")
    ax.set_title(f"frac($s_{{t-1}}$) $\\in$ [{fs_edges[bi]:.2f}, {fs_edges[bi+1]:.2f})\nN={mask.sum()}", fontsize=9)
    ax.legend(fontsize=7)
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)

fig.suptitle(f"Fig 2: Zone-only, binned by frac($s_{{t-1}}$) — {SLABEL}", y=1.02, fontsize=11)
fig.tight_layout(); plt.show()

### S1.3 Model comparison (zone only)

Compare candidate models restricted to the negotiation zone.

In [ ]:
models = [
    ("Raw linear",       np.column_stack([np.ones_like(m), m, s]), y, 3),
    ("Frac linear",      np.column_stack([np.ones_like(fm), fm, fs]), fy, 3),
    ("Frac + v",         np.column_stack([np.ones_like(fm), fm, fs, v]), fy, 4),
    ("Frac + t",         np.column_stack([np.ones_like(fm), fm, fs, ti]), fy, 4),
    ("Frac + fm*fs",     np.column_stack([np.ones_like(fm), fm, fs, fm*fs]), fy, 4),
    ("Frac quadratic",   np.column_stack([np.ones_like(fm), fm, fs, fm**2, fs**2, fm*fs]), fy, 6),
]

print(f"{'Model':<22} {'RMSE':>8} {'R2':>8} {'#P':>4}")
print("-"*46)
for name, X, yt, p in sorted(models, key=lambda x: ols(x[1],x[2])[1]):
    b, rmse, r2, _, _ = ols(X, yt)
    print(f"{name:<22} {rmse:>8.4f} {r2:>8.4f} {p:>4}")

### S1.4 Baseline coefficient table

$\text{frac}(s_t) = \beta_0 + \beta_m \cdot \text{frac}(m_t) + \beta_s \cdot \text{frac}(s_{t-1})$

In [ ]:
X_base = np.column_stack([np.ones_like(fm), fm, fs])
b, rmse, r2, yhat, se = ols(X_base, fy)
names = ["intercept", "frac(m_t)", "frac(s_prev)"]

print(f"Zone-only fractional linear: N={len(fy)}, RMSE={rmse:.4f}, R2={r2:.4f}")
print(f"\n  {'Feature':<16} {'Coef':>9} {'SE':>9} {'t-stat':>9}")
print(f"  {'---'*15}")
for i, name in enumerate(names):
    t_stat = b[i]/se[i] if se[i]>1e-8 else float('inf')
    sig = "***" if abs(t_stat)>3.29 else "**" if abs(t_stat)>2.58 else "*" if abs(t_stat)>1.96 else ""
    print(f"  {name:<16} {b[i]:>+9.4f} {se[i]:>9.4f} {t_stat:>+9.2f} {sig}")

print(f"\nInterpretation:")
print(f"  b(fm) + b(fs) = {b[1]+b[2]:.3f} (should be ~1.0 for convex combination)")
print(f"  The seller places {b[1]*100:.1f}% weight on buyer, {b[2]*100:.1f}% on own prev price")
print(f"  Intercept {b[0]:.3f} means a small upward bias of {b[0]*100:.1f}% of zone")

---
## S2. Time homogeneity (zone only)

Split zone transitions by round chunks, fit per-chunk, cross-predict.

### S2.1 Per-chunk fit

In [ ]:
all_rounds = sorted(set(ti))
n_chunks = min(6, max(2, len(all_rounds)//2))
chunk_edges = np.array_split(all_rounds, n_chunks)
chunks = [(int(c[0]), int(c[-1])) for c in chunk_edges]

chunk_models = {}
print(f"{'Chunk':<18} {'N':>5} {'b(fm)':>8} {'b(fs)':>8} {'sum':>6} {'RMSE':>8} {'R2':>8}")
print("-"*60)
for ci,(rlo,rhi) in enumerate(chunks):
    mask = (ti>=rlo)&(ti<=rhi)
    if mask.sum()<10: continue
    Xc = np.column_stack([np.ones(mask.sum()), fm[mask], fs[mask]])
    bc,rc,r2c,_,_ = ols(Xc, fy[mask])
    chunk_models[ci] = {"beta":bc, "mask":mask, "rounds":(rlo,rhi)}
    print(f"C{ci} (R{rlo}-{rhi}){'':<6} {mask.sum():>5} {bc[1]:>+8.3f} {bc[2]:>+8.3f} "
          f"{bc[1]+bc[2]:>6.3f} {rc:>8.4f} {r2c:>8.4f}")

### S2.2 Cross-prediction confusion matrix

In [ ]:
valid_chunks = sorted(chunk_models.keys())
nc = len(valid_chunks)

if nc >= 2:
    conf = np.full((nc, nc), np.nan)
    for i, ci in enumerate(valid_chunks):
        bi = chunk_models[ci]["beta"]
        for j, cj in enumerate(valid_chunks):
            mj = chunk_models[cj]["mask"]
            Xj = np.column_stack([np.ones(mj.sum()), fm[mj], fs[mj]])
            pred = Xj @ bi
            conf[i,j] = float(np.sqrt(np.mean((fy[mj]-pred)**2)))

    fig, ax = plt.subplots(figsize=(7, 5.5))
    im = ax.imshow(conf, cmap='YlOrRd', aspect='auto')
    plt.colorbar(im, ax=ax, label="RMSE (frac)")
    clabels = [f"R{chunk_models[c]['rounds'][0]}-{chunk_models[c]['rounds'][1]}" for c in valid_chunks]
    ax.set_xticks(range(nc)); ax.set_xticklabels(clabels, rotation=30, ha='right', fontsize=9)
    ax.set_yticks(range(nc)); ax.set_yticklabels(clabels, fontsize=9)
    ax.set_xlabel("Test chunk"); ax.set_ylabel("Train chunk")
    for i in range(nc):
        for j in range(nc):
            ax.text(j,i,f"{conf[i,j]:.3f}",ha='center',va='center',fontsize=8,
                    color='white' if conf[i,j]>np.nanmedian(conf) else 'black')
    ax.set_title(f"Fig 3: Cross-prediction RMSE (zone only) — {SLABEL}", fontsize=11)
    fig.tight_layout(); plt.show()

    diag = np.diag(conf); off = conf[~np.eye(nc,dtype=bool)]
    ratio = np.nanmean(off)/np.nanmean(diag)
    print(f"Diagonal mean:     {np.nanmean(diag):.4f}")
    print(f"Off-diagonal mean: {np.nanmean(off):.4f}")
    print(f"Ratio:             {ratio:.2f}x -> {'HOMOGENEOUS' if ratio<1.5 else 'STEP-DEPENDENT'}")
else:
    print("Not enough chunks for cross-prediction.")

### S2.3 Coefficient stability across chunks

In [ ]:
if nc >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ci_idx, cname in enumerate(['$\\beta_0$', '$\\beta_m$ (frac $m_t$)', '$\\beta_s$ (frac $s_{t-1}$)']):
        ax = axes[ci_idx]
        vals = [chunk_models[c]["beta"][ci_idx] for c in valid_chunks]
        clbls = [f"R{chunk_models[c]['rounds'][0]}-{chunk_models[c]['rounds'][1]}" for c in valid_chunks]
        ax.bar(range(len(vals)), vals, alpha=0.8)
        ax.set_xticks(range(len(vals)))
        ax.set_xticklabels(clbls, rotation=30, ha='right', fontsize=8)
        ax.set_title(cname); ax.axhline(0, color='gray', lw=0.5)
    fig.suptitle(f"Fig 4: Zone coefficients by chunk — {SLABEL}", y=1.02, fontsize=11)
    fig.tight_layout(); plt.show()

---
## S3. Covariate ablation (zone only)

Test whether adding extra features improves the zone-only fractional model.

### S3.1 Feature hierarchy

In [ ]:
mi_arr = np.array([t["m_init"] for t in trans])
si_arr = np.array([t["s_init"] for t in trans])
mp_arr = np.array([t["m_prev"] for t in trans])
sp2_raw = [t["s_prev2"] for t in trans]
has_sp2 = np.array([x is not None for x in sp2_raw])
sp2_arr = np.array([x if x is not None else 0.0 for x in sp2_raw])
fi_mp = (mp_arr - v) / zone
fi_mi = (mi_arr - v) / zone
fi_si = np.where(~np.isnan(si_arr), (si_arr - v) / zone, np.nan)
fi_sp2 = np.where(has_sp2, (sp2_arr - v) / zone, 0.0)
valid_si = ~np.isnan(fi_si)

X_base = np.column_stack([np.ones_like(fm), fm, fs])

ablations = [
    ("Baseline: frac(m_t) + frac(s_prev)",        X_base, fy, np.ones(len(fm),dtype=bool)),
    ("+ v (valuation)",                            np.column_stack([X_base,v]), fy, np.ones(len(fm),dtype=bool)),
    ("+ t (round index)",                          np.column_stack([X_base,ti]), fy, np.ones(len(fm),dtype=bool)),
    ("+ frac(m_prev)",                             np.column_stack([X_base,fi_mp]), fy, np.ones(len(fm),dtype=bool)),
    ("+ frac(m_init)",                             np.column_stack([X_base,fi_mi]), fy, np.ones(len(fm),dtype=bool)),
    ("+ frac(s_init)",                             np.column_stack([X_base[valid_si],fi_si[valid_si]]), fy[valid_si], valid_si),
    ("+ frac(s_prev2) [rounds>=3]",                np.column_stack([X_base[has_sp2],fi_sp2[has_sp2]]), fy[has_sp2], has_sp2),
    ("  (baseline restricted rounds>=3)",          X_base[has_sp2], fy[has_sp2], has_sp2),
    ("+ v + t + frac(m_prev)",                     np.column_stack([X_base,v,ti,fi_mp]), fy, np.ones(len(fm),dtype=bool)),
]

print(f"{'Model':<42} {'N':>5} {'RMSE':>8} {'R2':>8}")
print("-"*65)
for label, X, yt, mask in ablations:
    n = len(yt)
    if n < 10: continue
    b, rmse, r2, _, _ = ols(X, yt)
    print(f"{label:<42} {n:>5} {rmse:>8.4f} {r2:>8.4f}")

---
## S4. Cross-product stability

Fit the fractional model per product.  If coefficients are stable across
products, the fractional normalization successfully absorbs $v$.

### S4.1 Per-product coefficients

In [ ]:
print(f"{'Product':<28} {'N':>5} {'b0':>7} {'b(fm)':>8} {'b(fs)':>8} {'sum':>6} {'RMSE':>7} {'R2':>6}")
print("-"*78)
for key in ["product", "supplier_id", "buyer_type"]:
    groups = {}
    for idx, t in enumerate(trans):
        groups.setdefault(t[key], []).append(idx)
    for name, idxs in sorted(groups.items()):
        if len(idxs) < 10: continue
        mask = np.array(idxs)
        Xg = np.column_stack([np.ones(len(mask)), fm[mask], fs[mask]])
        bg, rg, r2g, _, _ = ols(Xg, fy[mask])
        print(f"  {str(name)[:26]:<28} {len(mask):>5} {bg[0]:>+7.3f} {bg[1]:>+8.3f} {bg[2]:>+8.3f} "
              f"{bg[1]+bg[2]:>6.3f} {rg:>7.4f} {r2g:>6.4f}")
    print()

---
## S5. Residual diagnostics (zone only)

### S5.1 True vs predicted and residual plots

In [ ]:
resid = fy - yhat

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# True vs pred (frac)
ax = axes[0,0]
lo = min(fy.min(),yhat.min())*0.9; hi = max(fy.max(),yhat.max())*1.1
ax.scatter(fy, yhat, alpha=0.4, s=10, edgecolors='none')
ax.plot([lo,hi],[lo,hi],'k--',lw=1)
ax.set_xlabel("True frac($s_t$)"); ax.set_ylabel("Predicted")
ax.set_title(f"True vs predicted (frac)  RMSE={rmse:.4f}")
ax.set_aspect('equal', adjustable='box')

# True vs pred (price)
ax = axes[0,1]
y_pred_price = v + zone * yhat
lo = min(y.min(),y_pred_price.min())*0.9; hi = max(y.max(),y_pred_price.max())*1.05
ax.scatter(y, y_pred_price, alpha=0.4, s=10, edgecolors='none')
ax.plot([lo,hi],[lo,hi],'k--',lw=1)
rmse_p = np.sqrt(np.mean((y-y_pred_price)**2))
ax.set_xlabel("True $s_t$ (\\$)"); ax.set_ylabel("Predicted (\\$)")
ax.set_title(f"True vs predicted (\\$)  RMSE=\\${rmse_p:.4f}")
ax.set_aspect('equal', adjustable='box')

# Residual vs frac(m_t)
ax = axes[1,0]
ax.scatter(fm, resid, alpha=0.3, s=8, edgecolors='none')
ax.axhline(0, color='red', ls='--', lw=1)
ax.set_xlabel("frac($m_t$)"); ax.set_ylabel("Residual")
ax.set_title("Residual vs frac($m_t$)")

# Residual by round
ax = axes[1,1]
rounds_u = sorted(set(ti))
means = [resid[ti==r].mean() for r in rounds_u if (ti==r).sum()>0]
stds = [resid[ti==r].std() for r in rounds_u if (ti==r).sum()>0]
valid_r = [int(r) for r in rounds_u if (ti==r).sum()>0]
ax.bar(valid_r, means, yerr=stds, capsize=2, alpha=0.7, edgecolor='black')
ax.axhline(0, color='red', ls='--', lw=1)
ax.set_xlabel("Round"); ax.set_ylabel("Mean residual")
ax.set_title("Residual by round (+-1 SD)")

fig.suptitle(f"Fig 5: Zone-only residual diagnostics — {SLABEL}", y=1.02, fontsize=11)
fig.tight_layout(); plt.show()

---
## S6. Summary for `cooperative` (zone only)

In [ ]:
print("=" * 65)
print(f"  ZONE-ONLY SUMMARY: {SLABEL}")
print("=" * 65)
print(f"  Zone transitions: {len(trans)} / {len(all_trans)} "
      f"({len(trans)/len(all_trans)*100:.0f}% of all data)")

b, rmse, r2, _, se = ols(np.column_stack([np.ones_like(fm),fm,fs]), fy)
print(f"  frac(s_t) = {b[0]:+.3f} + {b[1]:+.3f}*frac(m_t) + {b[2]:+.3f}*frac(s_prev)")
print(f"  RMSE = {rmse:.4f}, R2 = {r2:.4f}")
print(f"  b(fm) + b(fs) = {b[1]+b[2]:.3f}")
print(f"  Weight on buyer: {b[1]*100:.1f}%")
print(f"  Weight on own price: {b[2]*100:.1f}%")
print()

# Time homogeneity
if nc >= 2:
    print(f"  Time homogeneity: off/diag = {ratio:.2f}x")
print(f"  Covariates: adding v, t, m_prev gives <1% R2 improvement")